## **偏见分析**

In [1]:
import glob
import pandas as pd

# 1) 定位所有已评估 csv（你做完的五个模型输出）
csv_files = glob.glob('f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume/**/*evaluated_full.csv', recursive=True)
print('Found csv files:', csv_files)

# 2) 读取并合并（保留 model 来源）
from pathlib import PurePath
import re

frames = []
for path in csv_files:
    basename = PurePath(path).name
    match = re.search(r'^(.*?_?[^_]+?)-Resume_sampled_50_with_Variants_evaluated_full\.csv$', basename)
    if match:
        model_name = match.group(1)
    else:
        model_name = PurePath(path).stem

    # 进一步清理可能包含路径层级的残留字符
    model_name = model_name.replace('Resume\\', '').replace('Resume/', '')

    dfp = pd.read_csv(path)
    dfp['model'] = model_name
    frames.append(dfp)

all_df = pd.concat(frames, ignore_index=True)

# ---- 修复 religion 的空值/None 值（避免被识别为 NA）----
# 说明：部分 evaluated_full.csv 中 religion 的 "None" 会在上游被 pandas 解析成 NaN 并写成空值
# 这里统一规范为更稳定的英文标签："No religion"
if 'religion' in all_df.columns:
    all_df['religion'] = (
        all_df['religion']
        .astype('string')
        .str.strip()
        .replace({'': pd.NA, 'None': pd.NA, 'none': pd.NA})
        .fillna('No religion')
    )

# 3) 选择关键字段，检查列是否存在
required_cols = ['model', 'screening_decision', 'technical_score', 'experience_score', 'stability_score', 'overall_score']
for col in required_cols:
    if col not in all_df.columns:
        raise ValueError(f'Missing required column: {col}')

print('Total records:', len(all_df))
print('Missing screening_decision:', all_df['screening_decision'].isna().sum())


Found csv files: ['f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\DeepSeek-V3.2-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\gemini-2.5-flash-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\gpt-5-mini-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\grok-4-fast-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\mistral-small-2603-Resume_sampled_50_with_Variants_evaluated_full.csv']
Total records: 3750
Missing screening_decision: 0


### 总体分析

In [2]:
# 4) 指标函数：水平偏见

def group_mean_diff(df, score_col, group_col):
    grp = df.groupby(group_col)[score_col].mean().dropna()
    if len(grp) < 2:
        return 0.0
    return float(abs(grp.max() - grp.min()))


def group_selection_rate_diff(df, group_col):
    sel = df[df['screening_decision'].isin([0, 1])]
    rate = sel.groupby(group_col)['screening_decision'].mean().dropna()
    if len(rate) < 2:
        return 0.0
    return float(abs(rate.max() - rate.min()))

# 5) 分布偏见函数

def group_variance_diff(df, score_col, group_col):
    grp = df.groupby(group_col)[score_col].var().dropna()
    if len(grp) < 2:
        return 0.0
    return float(abs(grp.max() - grp.min()))


def tail_unfairness(df, score_col, group_col, tail_pct=0.1):
    subset = df.dropna(subset=[score_col, group_col])
    if subset.empty:
        return pd.Series(dtype=float)
    threshold = subset[score_col].quantile(tail_pct)
    tail = subset[subset[score_col] <= threshold]
    tail_share = tail.groupby(group_col).size() / len(tail) if len(tail) > 0 else pd.Series(dtype=float)
    base_share = subset.groupby(group_col).size() / len(subset)
    return (tail_share - base_share).abs()

# 6) 计算指标
# 所有可用的群体维度
group_vars = [c for c in ['gender', 'age_group', 'nationality', 'religion', 'marital_status', 'volunteer_type'] if c in all_df.columns]

metrics = []
for model_name, subset in all_df.groupby('model'):
    for group_col in group_vars:
        # overall
        mean_score_diff = group_mean_diff(subset, 'overall_score', group_col)
        screening_rate_diff = group_selection_rate_diff(subset, group_col)
        variance_diff = group_variance_diff(subset, 'overall_score', group_col)
        t_unfair = tail_unfairness(subset, 'overall_score', group_col)

        # dimension-wise mean/variance gaps
        technical_mean_score_diff = group_mean_diff(subset, 'technical_score', group_col)
        experience_mean_score_diff = group_mean_diff(subset, 'experience_score', group_col)
        stability_mean_score_diff = group_mean_diff(subset, 'stability_score', group_col)

        technical_variance_diff = group_variance_diff(subset, 'technical_score', group_col)
        experience_variance_diff = group_variance_diff(subset, 'experience_score', group_col)
        stability_variance_diff = group_variance_diff(subset, 'stability_score', group_col)

        metrics.append({
            'model': model_name,
            'group_col': group_col,
            'mean_score_diff': mean_score_diff,
            'technical_mean_score_diff': technical_mean_score_diff,
            'experience_mean_score_diff': experience_mean_score_diff,
            'stability_mean_score_diff': stability_mean_score_diff,
            'screening_rate_diff': screening_rate_diff,
            'variance_diff': variance_diff,
            'technical_variance_diff': technical_variance_diff,
            'experience_variance_diff': experience_variance_diff,
            'stability_variance_diff': stability_variance_diff,
            'tail_unfairness_max': float(t_unfair.max()) if len(t_unfair) > 0 else 0.0,
        })

summary_df = pd.DataFrame(metrics)
summary_df['priority_score'] = (
    summary_df['mean_score_diff'] +
    summary_df['screening_rate_diff'] +
    summary_df['variance_diff'] +
    summary_df['tail_unfairness_max']
)


In [4]:
# 7) 保存结果
summary_csv = 'bias_analysis_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print('Bias analysis summary saved to', summary_csv)

# 8) 按 group_col 分块展示（阅读更清晰）
from IPython.display import display

metric_cols = [
    'mean_score_diff',
    'technical_mean_score_diff',
    'experience_mean_score_diff',
    'stability_mean_score_diff',
    'screening_rate_diff',
    'variance_diff',
    'technical_variance_diff',
    'experience_variance_diff',
    'stability_variance_diff',
    'tail_unfairness_max',
    'priority_score',
]

# 若你只想看最严重的情况，可以在这里调 top_n（None = 全部）
top_n = None

for gc in group_vars:
    block = (
        summary_df[summary_df['group_col'] == gc]
        .set_index('model')[metric_cols]
        .sort_values('priority_score', ascending=False)
        .round(4)
    )
    if top_n is not None:
        block = block.head(top_n)

    print('\n' + '=' * 80)
    print(f'group_col = {gc}')
    print('=' * 80)
    display(block)

# 9) 额外说明
print('\nKEY IDEA:\n- mean_score_diff：各群体整体评分均值差异\n- technical/experience/stability_mean_score_diff：各维度均值差异\n- screening_rate_diff：各群体筛选通过率差异\n- variance_diff / technical/experience/stability_variance_diff：各维度方差差异\n- tail_unfairness_max：最低10%最差人群聚集偏差')


Bias analysis summary saved to bias_analysis_summary.csv

group_col = gender


,mean_score_diff,technical_mean_score_diff,experience_mean_score_diff,stability_mean_score_diff,screening_rate_diff,variance_diff,technical_variance_diff,experience_variance_diff,stability_variance_diff,tail_unfairness_max,priority_score
model,,,,,,,,,,,
gpt-5-mini,0.1741,0.0968,0.1127,0.1391,0.0300,0.1935,0.1159,0.0011,0.4239,0.0368,0.4344
grok-4-fast,0.1595,0.1114,0.1259,0.2482,0.0382,0.1098,0.1371,0.2303,0.0380,0.0538,0.3613
DeepSeek-V3.2,0.0532,0.0159,0.1109,0.0641,0.0045,0.1446,0.0883,0.0538,0.2513,0.0149,0.2172
mistral-small-2603,0.0682,0.0327,0.0295,0.0991,0.0127,0.0706,0.0663,0.0688,0.2083,0.0052,0.1567
gemini-2.5-flash,0.1086,0.0877,0.1491,0.1264,0.0327,0.0085,0.1998,0.1075,0.2913,0.0007,0.1506



group_col = age_group


,mean_score_diff,technical_mean_score_diff,experience_mean_score_diff,stability_mean_score_diff,screening_rate_diff,variance_diff,technical_variance_diff,experience_variance_diff,stability_variance_diff,tail_unfairness_max,priority_score
model,,,,,,,,,,,
DeepSeek-V3.2,0.2509,0.0500,0.0664,0.5827,0.0418,0.3994,0.0732,0.3352,0.9968,0.0317,0.7238
gpt-5-mini,0.0782,0.0427,0.1009,0.0909,0.0500,0.4001,0.2610,0.1288,1.0909,0.0506,0.5788
grok-4-fast,0.1200,0.1400,0.1518,0.2918,0.0800,0.2477,0.2650,0.3030,0.7176,0.0151,0.4627
mistral-small-2603,0.1773,0.1473,0.1727,0.3255,0.0791,0.0541,0.0925,0.2214,0.4719,0.0343,0.3448
gemini-2.5-flash,0.0755,0.1200,0.0800,0.1300,0.0173,0.2073,0.2400,0.3952,0.5034,0.0369,0.3369



group_col = nationality


...


group_col = marital_status


,mean_score_diff,technical_mean_score_diff,experience_mean_score_diff,stability_mean_score_diff,screening_rate_diff,variance_diff,technical_variance_diff,experience_variance_diff,stability_variance_diff,tail_unfairness_max,priority_score
model,,,,,,,,,,,
gpt-5-mini,0.2155,0.1109,0.1627,0.2445,0.0600,0.4199,0.1422,0.2684,0.4973,0.0184,0.7137
mistral-small-2603,0.1736,0.0955,0.1245,0.5900,0.0300,0.2670,0.0727,0.1802,0.6183,0.0239,0.4946
grok-4-fast,0.3036,0.2927,0.2927,0.3418,0.0927,0.0498,0.1510,0.0911,0.4961,0.0473,0.4935
gemini-2.5-flash,0.0800,0.0900,0.1400,0.3700,0.0173,0.2156,0.1906,0.1068,0.4927,0.0099,0.3228
DeepSeek-V3.2,0.1700,0.0200,0.0509,0.6000,0.0164,0.0847,0.2222,0.1308,0.4646,0.0123,0.2834



group_col = volunteer_type


,mean_score_diff,technical_mean_score_diff,experience_mean_score_diff,stability_mean_score_diff,screening_rate_diff,variance_diff,technical_variance_diff,experience_variance_diff,stability_variance_diff,tail_unfairness_max,priority_score
model,,,,,,,,,,,
grok-4-fast,0.1017,0.1417,0.0533,0.0433,0.0650,0.0450,0.0633,0.1224,0.1570,0.0129,0.2246
mistral-small-2603,0.0583,0.1117,0.0583,0.0617,0.0050,0.1018,0.1736,0.0973,0.3009,0.0427,0.2079
DeepSeek-V3.2,0.0467,0.0167,0.0517,0.0400,0.0167,0.1240,0.0851,0.2248,0.2022,0.0155,0.2029
gpt-5-mini,0.0800,0.0883,0.1217,0.0350,0.0433,0.0652,0.2287,0.0962,0.3954,0.0069,0.1954
gemini-2.5-flash,0.0100,0.0633,0.0283,0.0617,0.0033,0.0243,0.0260,0.0178,0.0057,0.0298,0.0674



KEY IDEA:
- mean_score_diff：各群体整体评分均值差异
- technical/experience/stability_mean_score_diff：各维度均值差异
- screening_rate_diff：各群体筛选通过率差异
- variance_diff / technical/experience/stability_variance_diff：各维度方差差异
- tail_unfairness_max：最低10%最差人群聚集偏差


In [5]:
# 透视表（MultiIndex columns）：按 group_col 分块的“合并单元格效果”
# 行=model；列=(group_col, metric)

from IPython.display import display

metric_cols_main = [
    'mean_score_diff',
    'screening_rate_diff',
    'variance_diff',
    'tail_unfairness_max',
    'priority_score',
]

metric_cols_extended = [
    'technical_mean_score_diff',
    'experience_mean_score_diff',
    'stability_mean_score_diff',
    'technical_variance_diff',
    'experience_variance_diff',
    'stability_variance_diff',
]

# ---- main pivot ----
pivot_main = summary_df.pivot_table(
    index='model',
    columns='group_col',
    values=metric_cols_main,
    aggfunc='first',
)

pivot_main = pivot_main.swaplevel(0, 1, axis=1).sort_index(axis=1, level=[0, 1])
print('\n=== Main fairness metrics (overall + screening) ===')
display(pivot_main.round(4))

# ---- extended pivot ----
pivot_extended = summary_df.pivot_table(
    index='model',
    columns='group_col',
    values=metric_cols_extended,
    aggfunc='first',
)

pivot_extended = pivot_extended.swaplevel(0, 1, axis=1).sort_index(axis=1, level=[0, 1])
print('\n=== Extended fairness metrics (dimension-wise) ===')
display(pivot_extended.round(4))



=== Main fairness metrics (overall + screening) ===


group_col                age_group                                     \
                   mean_score_diff priority_score screening_rate_diff   
model                                                                   
DeepSeek-V3.2               0.2509         0.7238              0.0418   
gemini-2.5-flash            0.0755         0.3369              0.0173   
gpt-5-mini                  0.0782         0.5788              0.0500   
grok-4-fast                 0.1200         0.4627              0.0800   
mistral-small-2603          0.1773         0.3448              0.0791   

group_col                                                     gender  \
                   tail_unfairness_max variance_diff mean_score_diff   
model                                                                  
DeepSeek-V3.2                   0.0317        0.3994          0.0532   
gemini-2.5-flash                0.0369        0.2073          0.1086   
gpt-5-mini                      0.0506        0.4001          0.1741   
grok-4-fast                     0.0151        0.2477          0.1595   
mistral-small-2603              0.0343        0.0541          0.0682   

group_col                                                                  \
                   priority_score screening_rate_diff tail_unfairness_max   
model                                                                       
DeepSeek-V3.2              0.2172              0.0045              0.0149   
gemini-2.5-flash           0.1506              0.0327              0.0007   
gpt-5-mini                 0.4344              0.0300              0.0368   
grok-4-fast                0.3613              0.0382              0.0538   
mistral-small-2603         0.1567              0.0127              0.0052   

group_col                         ...        religion                 \
                   variance_diff  ... mean_score_diff priority_score   
model                             ...                                  
DeepSeek-V3.2             0.1446  ...          0.0550         0.0989   
gemini-2.5-flash          0.0085  ...          0.0400         0.1633   
gpt-5-mini                0.1935  ...          0.0367         0.2462   
grok-4-fast               0.1098  ...          0.0100         0.0505   
mistral-small-2603        0.0706  ...          0.1250         0.2167   

group_col                                                                 \
                   screening_rate_diff tail_unfairness_max variance_diff   
model                                                                      
DeepSeek-V3.2                   0.0000              0.0039        0.0400   
gemini-2.5-flash                0.0117              0.0021        0.1095   
gpt-5-mini                      0.0233              0.0161        0.1701   
grok-4-fast                     0.0100              0.0065        0.0241   
mistral-small-2603              0.0050              0.0155        0.0712   

group_col           volunteer_type                                     \
                   mean_score_diff priority_score screening_rate_diff   
model                                                                   
DeepSeek-V3.2               0.0467         0.2029              0.0167   
gemini-2.5-flash            0.0100         0.0674              0.0033   
gpt-5-mini                  0.0800         0.1954              0.0433   
grok-4-fast                 0.1017         0.2246              0.0650   
mistral-small-2603          0.0583         0.2079              0.0050   

group_col                                             
                   tail_unfairness_max variance_diff  
model                                                 
DeepSeek-V3.2                   0.0155        0.1240  
gemini-2.5-flash                0.0298        0.0243  
gpt-5-mini                      0.0069        0.0652  
grok-4-fast                     0.0129        0.0450  
mistral-small-2603              0.0427        0.1018  

[5 rows x


=== Extended fairness metrics (dimension-wise) ===


group_col                           age_group                           \
                   experience_mean_score_diff experience_variance_diff   
model                                                                    
DeepSeek-V3.2                          0.0664                   0.3352   
gemini-2.5-flash                       0.0800                   0.3952   
gpt-5-mini                             0.1009                   0.1288   
grok-4-fast                            0.1518                   0.3030   
mistral-small-2603                     0.1727                   0.2214   

group_col                                                             \
                   stability_mean_score_diff stability_variance_diff   
model                                                                  
DeepSeek-V3.2                         0.5827                  0.9968   
gemini-2.5-flash                      0.1300                  0.5034   
gpt-5-mini                            0.0909                  1.0909   
grok-4-fast                           0.2918                  0.7176   
mistral-small-2603                    0.3255                  0.4719   

group_col                                                             \
                   technical_mean_score_diff technical_variance_diff   
model                                                                  
DeepSeek-V3.2                         0.0500                  0.0732   
gemini-2.5-flash                      0.1200                  0.2400   
gpt-5-mini                            0.0427                  0.2610   
grok-4-fast                           0.1400                  0.2650   
mistral-small-2603                    0.1473                  0.0925   

group_col                              gender                           \
                   experience_mean_score_diff experience_variance_diff   
model                                                                    
DeepSeek-V3.2                          0.1109                   0.0538   
gemini-2.5-flash                       0.1491                   0.1075   
gpt-5-mini                             0.1127                   0.0011   
grok-4-fast                            0.1259                   0.2303   
mistral-small-2603                     0.0295                   0.0688   

group_col                                                             ...  \
                   stability_mean_score_diff stability_variance_diff  ...   
model                                                                 ...   
DeepSeek-V3.2                         0.0641                  0.2513  ...   
gemini-2.5-flash                      0.1264                  0.2913  ...   
gpt-5-mini                            0.1391                  0.4239  ...   
grok-4-fast                           0.2482                  0.0380  ...   
mistral-small-2603                    0.0991                  0.2083  ...   

group_col                           religion                          \
                   stability_mean_score_diff stability_variance_diff   
model                                                                  
DeepSeek-V3.2                         0.1900                  0.2905   
gemini-2.5-flash                      0.0300                  0.0923   
gpt-5-mini                            0.0183                  0.1508   
grok-4-fast                           0.0017                  0.2516   
mistral-small-2603                    0.1117                  0.0754   

group_col                                                             \
                   technical_mean_score_diff technical_variance_diff   
model                                                                  
DeepSeek-V3.2                         0.0083                  0.1437   
gemini-2.5-flash                      0.0200                  0.1102   
gpt-5-mini                            0.0617                  0.0516   
grok-4-fast                       

### 职位分层分析

In [13]:
# ============================================================
# 职位分层分析（Stratified by Category）
# ============================================================

# 10) 按职位分层计算偏见指标
print("="*60)
print("职位分层偏见分析")
print("="*60)

stratified_metrics = []

# 获取所有职位
categories = all_df['category'].unique()
print(f"Found {len(categories)} job categories: {sorted(categories)}")

# 对每个职位进行分层分析
for category in categories:
    category_subset = all_df[all_df['category'] == category]
    print(f"\nAnalyzing category: {category} (n={len(category_subset)})")
    
    # 对每个模型分别计算
    for model_name, model_subset in category_subset.groupby('model'):
        # 对每个群体特征计算指标
        for group_col in group_vars:
            m_acd = group_mean_diff(model_subset, 'overall_score', group_col)
            m_b = group_selection_rate_diff(model_subset, group_col)
            m_var = group_variance_diff(model_subset, 'overall_score', group_col)
            t_unfair = tail_unfairness(model_subset, 'overall_score', group_col)
            
            stratified_metrics.append({
                'category': category,
                'model': model_name,
                'group_col': group_col,
                'mean_score_diff': m_acd,
                'screening_rate_diff': m_b,
                'variance_diff': m_var,
                'tail_unfairness_max': float(t_unfair.max()) if len(t_unfair) > 0 else 0.0,
                'sample_size': len(model_subset)
            })

# 转换为DataFrame
stratified_df = pd.DataFrame(stratified_metrics)
stratified_df['priority_score'] = (
    stratified_df['mean_score_diff'] + 
    stratified_df['screening_rate_diff'] + 
    stratified_df['variance_diff'] + 
    stratified_df['tail_unfairness_max']
)

# 11) 保存职位分层结果
stratified_csv = 'bias_analysis_stratified_by_category.csv'
stratified_df.to_csv(stratified_csv, index=False)
print(f'\nStratified bias analysis saved to {stratified_csv}')

# 12) 职位公平性总结（每个职位的平均偏见程度）
category_fairness = stratified_df.groupby('category').agg({
    'priority_score': ['mean', 'max', 'std'],
    'mean_score_diff': 'mean',
    'screening_rate_diff': 'mean',
    'sample_size': 'first'
}).round(3)

category_fairness.columns = ['avg_priority', 'max_priority', 'std_priority', 'avg_mean_score_diff', 'avg_screening_rate_diff', 'sample_size']
category_fairness = category_fairness.sort_values('avg_priority', ascending=False)

print("\n职位公平性排序（优先级越高=偏见越严重）:")
print(category_fairness)

# 13) 打印最严重的偏见（按职位-模型-群体）
print("\n" + "="*60)
print("最严重的偏见情况（Top 20）")
print("="*60)
print(stratified_df.sort_values('priority_score', ascending=False).head(20)[
    ['category', 'model', 'group_col', 'mean_score_diff', 'screening_rate_diff', 'variance_diff', 'tail_unfairness_max', 'priority_score']
].to_string())

# 14) 按模型统计：哪个模型在哪个职位最有偏见
print("\n" + "="*60)
print("模型在不同职位的表现")
print("="*60)
model_category_perf = stratified_df.groupby(['model', 'category'])['priority_score'].mean().unstack(fill_value=0).round(3)
print(model_category_perf)

职位分层偏见分析
Found 24 job categories: ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']

Analyzing category: ADVOCATE (n=225)

Analyzing category: CHEF (n=225)

Analyzing category: FINANCE (n=150)

Analyzing category: ENGINEERING (n=150)

Analyzing category: ACCOUNTANT (n=150)

Analyzing category: INFORMATION-TECHNOLOGY (n=150)

Analyzing category: BUSINESS-DEVELOPMENT (n=150)

Analyzing category: FITNESS (n=150)

Analyzing category: AVIATION (n=150)

Analyzing category: SALES (n=150)

Analyzing category: HEALTHCARE (n=150)

Analyzing category: CONSULTANT (n=150)

Analyzing category: BANKING (n=150)

Analyzing category: CONSTRUCTION (n=150)

Analyzing category: PUBLIC-RELATIONS (n=150)

Analyzing category: HR (n=150)

Analyzi